In [1]:
import sys
from pathlib import Path

# Add project root to Python path
sys.path.append(str(Path().resolve().parent))

# Test — DrugSafetyHGNN (encoder + decoder glue)

Tests `models/hgnn.py` against the real graph object from Step 1/2.

In [2]:
import torch

from models.hgnn import DrugSafetyHGNN
from models.encoder import build_message_passing_edges, split_polypharmacy_edges
from models.decoder import DistMultDecoder

PROJECT_ROOT = Path("..")

GRAPH_DIR = PROJECT_ROOT / "graph"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
data = torch.load(GRAPH_DIR / "heterodata_with_features.pt", weights_only=False)

ddi_key = ("drug", "polypharmacy", "drug")
num_relations = int(data[ddi_key].edge_type.max().item()) + 1
print("num_relations:", num_relations)
print(data)


c:\Users\sth3ayush\Desktop\Hackathon\IIMS x Perceptron 2026\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


num_relations: 1301
HeteroData(
  drug={
    num_nodes=70,
    x=[70, 2048],
  },
  gene={ num_nodes=19083 },
  side_effect={
    num_nodes=5004,
    disease_class=[5004],
    num_disease_classes=1,
  },
  (gene, interacts, gene)={ edge_index=[2, 1431224] },
  (drug, targets, gene)={ edge_index=[2, 2522] },
  (gene, targeted_by, drug)={ edge_index=[2, 2522] },
  (drug, causes, side_effect)={ edge_index=[2, 20345] },
  (side_effect, caused_by, drug)={ edge_index=[2, 20345] },
  (drug, polypharmacy, drug)={
    edge_index=[2, 801630],
    edge_type=[801630],
  }
)


## Test 1 — missing `num_relations` fails loudly at construction

In [3]:
try:
    DrugSafetyHGNN(data)
    print("FAIL -- no error raised")
except AssertionError as e:
    print("PASS -- raised clear AssertionError:")
    print(" ", e)


PASS -- raised clear AssertionError:
  num_relations must be provided (e.g. len(relation_vocab) or via DistMultDecoder.from_relation_vocab)


## Test 2 — default forward pass (safe: no DDI edges in message passing)

In [4]:
torch.manual_seed(0)
model = DrugSafetyHGNN(data, num_relations=num_relations)

sample_edge_index = data[ddi_key].edge_index[:, :5]
sample_edge_type = data[ddi_key].edge_type[:5]

scores = model(data, sample_edge_index, sample_edge_type)

assert scores.shape == (5,)
print("scores shape:", scores.shape)
print("scores:", scores)
print("PASS")


scores shape: torch.Size([5])
scores: tensor([-0.0036, -0.0036, -0.0459, -0.0459,  0.0288], grad_fn=<SumBackward1>)
PASS


## Test 3 — leakage check

In [5]:
torch.manual_seed(1)
model = DrugSafetyHGNN(data, num_relations=num_relations)
model.eval()

edge_index = data[ddi_key].edge_index
edge_type = data[ddi_key].edge_type

# score edge 1 while edge 0 is present
target_edge_index = edge_index[:, 1:2]
target_edge_type = edge_type[1:2]

with torch.no_grad():
    score_before = model(data, target_edge_index, target_edge_type).clone()

    # remove a DIFFERENT edge (edge 0) from the graph entirely
    src0, dst0 = edge_index[0, 0].item(), edge_index[1, 0].item()
    mask = ~(((edge_index[0] == src0) & (edge_index[1] == dst0)) |
             ((edge_index[0] == dst0) & (edge_index[1] == src0)))
    data_ablated = data.clone()
    data_ablated[ddi_key].edge_index = edge_index[:, mask]
    data_ablated[ddi_key].edge_type = edge_type[mask]

    score_after = model(data_ablated, target_edge_index, target_edge_type)

diff = (score_before - score_after).abs().item()
print(f"score diff after removing an unrelated edge: {diff:.8f}")
assert diff == 0.0
print("PASS -- unrelated edge removal has zero effect on this edge's score")


score diff after removing an unrelated edge: 0.00000000
PASS -- unrelated edge removal has zero effect on this edge's score


## Test 4 — explicit train-split message passing, scoring held-out val edges

In [6]:
torch.manual_seed(2)
splits = split_polypharmacy_edges(data, val_frac=0.1, test_frac=0.1, seed=42)

mp_edges = build_message_passing_edges(data)

# FIX: sample a small, fixed-size subset of train edges for message passing
# instead of merging the entire train split. The full split can be millions
# of edges at real scale -- exactly the pattern that OOM'd during actual
# training. This test only needs to prove the override plumbing works.
train_edge_index = splits["train"]["edge_index"]
sample_size = min(1000, train_edge_index.shape[1])
sample_idx = torch.randperm(train_edge_index.shape[1])[:sample_size]
mp_edges[ddi_key] = train_edge_index[:, sample_idx]

model = DrugSafetyHGNN(data, num_relations=num_relations)

val_scores = model(
    data,
    splits["val"]["edge_index"],
    splits["val"]["edge_type"],
    message_passing_edges=mp_edges,
)

print("train edges available:", train_edge_index.shape[1], "| sampled for message passing:", sample_size)
print("val edges:", splits["val"]["edge_index"].shape[1])
print("val scores shape:", val_scores.shape)
assert val_scores.shape[0] == splits["val"]["edge_index"].shape[1]
print("PASS")


train edges available: 641304 | sampled for message passing: 1000
val edges: 80163
val scores shape: torch.Size([80163])
PASS


## Test 5 — decoder inside the model matches a standalone decoder built from the real relation lookup

In [7]:
relation_vocab_path = PROCESSED_DIR / "polypharmacy_relation_lookup.csv"
standalone_decoder = DistMultDecoder.from_relation_vocab(relation_vocab_path)

assert model.decoder.num_relations == standalone_decoder.num_relations == num_relations
print("model.decoder.num_relations:", model.decoder.num_relations)
print("PASS -- matches the real relation vocabulary size")


model.decoder.num_relations: 1301
PASS -- matches the real relation vocabulary size


## Test 6 — gradients actually flow end-to-end

In [8]:
model = DrugSafetyHGNN(data, num_relations=num_relations)
model.train()

scores = model(data, sample_edge_index, sample_edge_type)
loss = scores.sum()
loss.backward()

encoder_grad = model.encoder.drug_projection.weight.grad
decoder_grad = model.decoder.relation_embedding.weight.grad

assert encoder_grad is not None and encoder_grad.abs().sum().item() > 0
assert decoder_grad is not None and decoder_grad.abs().sum().item() > 0
print("PASS -- gradients reached both encoder.drug_projection and decoder.relation_embedding")


PASS -- gradients reached both encoder.drug_projection and decoder.relation_embedding
